In [4]:
!pip install biopython

In [20]:
import gzip
from Bio import SeqIO
from typing import List

In [10]:
fasta_file = r"C:\Users\carol\Downloads\human_g1k_v37.fasta.gz"

with gzip.open(fasta_file, "rt") as f:
    print(f.readline())  # Read just the first line


>1 dna:chromosome chromosome:GRCh37:1:1:249250621:1



In [6]:
with open(fasta_file, "rb") as f:
    print(f.read(2))

b'\x1f\x8b'


In [7]:
import os
print(os.path.getsize(fasta_file))  # Should be a big number


892331003


In [21]:
#Troubleshooted with ChatGPT gzip file error. Used their suggested way of only parsing out the chromosome
def extract_15mers_from_chr1(fasta_path):
    fifteen_mers = []

    with gzip.open(fasta_path, "rt") as handle:
        for record in SeqIO.parse(handle, "fasta"):
            if record.id.strip().lower() in ["1", "chr1"]:
                sequence = str(record.seq)
                for i in range(len(sequence) - 14):  # 15-mers
                    fifteen_mers.append(sequence[i:i+15])
                return fifteen_mers

# File path
fasta_file = r"C:\Users\carol\Downloads\human_g1k_v37.fasta.gz"

# Run the extraction
kmers = extract_15mers_from_chr1(fasta_file)

# Check a few
print(kmers[:5])


['NNNNNNNNNNNNNNN', 'NNNNNNNNNNNNNNN', 'NNNNNNNNNNNNNNN', 'NNNNNNNNNNNNNNN', 'NNNNNNNNNNNNNNN']


In [22]:
len(kmers)*15

3738759105

In [24]:
#asked ChatGPT how to create a for loop that would drop any kmers that meet given criteria

filtered_kmers = []
for kmer in kmers:
    if kmer.count('N') <= 2:
        filtered_kmers.append(kmer)

len(filtered_kmers)

225280241

In [12]:
# visualize first 5 filtered kmers

print(filtered_kmers[:5])

['NNTAACCCTAACCCT', 'NTAACCCTAACCCTA', 'TAACCCTAACCCTAA', 'AACCCTAACCCTAAC', 'ACCCTAACCCTAACC']


In [25]:
# join all kmers into one string so it is useable by the rolling hash
kmers_string = ''.join(filtered_kmers)

In [26]:
# visualize to ensure joined 
print(kmers_string[:20])

NNTAACCCTAACCCTNTAAC


In [9]:
#test code on synthetic data before running on chromosome 1
# create fake data
fake_mers = ['AGTCA', 'TGCAT','CGCGT', 'AGTCA', 'GACTA']

In [10]:
#fake string
fake_string = ''.join(fake_mers)

In [ ]:
# source: https://www.geeksforgeeks.org/dsa/introduction-to-rolling-hash-data-structures-and-algorithms/
# Python Implementation
# is this what was killing my computer? because too many hash values to return

from typing import List


# def rolling_hash(s: str, window_size: int, base: int = 26, mod: int = 10**9 + 7) -> List[int]:
    """
    Calculates the rolling hash values of all substrings of length window_size in string s.
    Uses the polynomial rolling hash algorithm with base and mod as constants.

    :param s: the input string
    :param window_size: the size of the rolling window
    :param base: the base for the polynomial hash function
    :param mod: the modulus for the polynomial hash function
    :return: a list of hash values of all substrings of length window_size in s
    """
    n = len(s)
    power = [1] * (n + 1)
    hash_values = [0] * (n - window_size + 1)

    # Precompute the powers of the
    # base modulo the mod
    for i in range(1, n + 1):
        power[i] = (power[i - 1] * base) % mod

    # Compute the hash value of
    # the first window
    current_hash = 0
    for i in range(window_size):
        current_hash = (current_hash * base + ord(s[i])) % mod

    hash_values[0] = current_hash

    # Compute the hash values of the
    # rest of the substrings
    for i in range(1, n - window_size + 1):

        # Remove the contribution of the
        # first character in the window
        current_hash = (
            current_hash - power[window_size - 1] * ord(s[i - 1])) % mod

        # Shift the window by one character
        # and add the new character
        # to the hash
        current_hash = (current_hash * base + ord(s[i + window_size - 1])) % mod

        hash_values[i] = current_hash

    return hash_values

In [ ]:
# Compute hashes for all 15-mers efficiently using rolling updates 
# to change base, input new when calling

# Driver code
# def main():

    # Input string
    s = fake_string

    # Window size
    window_size = 15

    # Calculate rolling hash values
    hash_values = rolling_hash(s, window_size)

    # Print the hash values
    print(hash_values)
    
if __name__ == "__main__":
    main()

[406034920, 367156408, 892645743, 717418068, 641895080, 499520526, 496162483, 246803659, 405920530, 364182287, 977368291]


In [ ]:
# Generate multiple independent hash functions by varying base a
# For each hash function, track minimum value seen
# Normalize to between 0-1 by dividing by M
# Combine results across hash functions - mean of minima
# Convert result into estimate of the number of distinct 15 mers
minimum_hash = []
all_normalized_min = []

# def main_test_bases():
    window_size = 15
    bases_to_test = [1, 2, 5, 10, 26, 100]
    for base in bases_to_test:
        hash_values = rolling_hash(fake_string, window_size, base=base)
        min_hash = min(hash_values) if hash_values else None
        minimum_hash.append(min_hash)
        M = 10**9 + 7
        normalized_min = min_hash/ M
        all_normalized_min.append(normalized_min)
        print(f"Base {base}: normalized minimum: {normalized_min}, minimum hash {min_hash} All: {hash_values}")
    combined_result = sum(all_normalized_min)/len(all_normalized_min)
    print(f"mean of minima across hash funcations {combined_result}")
    estimated_distinct_kmers = (1 / combined_result) - 1
    print(f"estimate of distinct 15-mers based on all hashes:{estimated_distinct_kmers}")

if __name__ == '__main__':
    main_test_bases()

Base 1: normalized minimum: 1.063999992552e-06, minimum hash 1064 All: [1083, 1083, 1083, 1083, 1083, 1083, 1070, 1064, 1064, 1083, 1064]
Base 2: normalized minimum: 0.002255948984208357, minimum hash 2255949 All: [2273174, 2416493, 2506529, 2260630, 2325871, 2521887, 2291333, 2256203, 2317017, 2504198, 2255949]
Base 5: normalized minimum: 0.01085868292398922, minimum hash 10858683 All: [10858683, 411729243, 310614566, 76528348, 704921747, 882044542, 933678201, 920359336, 924076659, 977819114, 412551048]
Base 10: normalized minimum: 0.07823886645232793, minimum hash 78238867 All: [793038407, 385384079, 350840833, 96408386, 433083920, 785839237, 446392385, 960923887, 78238867, 237388747, 961887521]
Base 26: normalized minimum: 0.2468036572723744, minimum hash 246803659 All: [406034920, 367156408, 892645743, 717418068, 641895080, 499520526, 496162483, 246803659, 405920530, 364182287, 977368291]
Base 100: normalized minimum: 0.087346285388576, minimum hash 87346286 All: [803243499, 346644

In [ ]:
# Attempt to use rolling hash to avoid storing all 15 distinct mers
# Asked ChatGPT how to analyze the distinct 15mers without exceeding my computer memory 
import gzip
from Bio import SeqIO

def estimate_distinct_15mers(fasta_file, base, M=10**9 + 7):
    min_hash = None
    window_size = 15

    with gzip.open(fasta_file, "rt") as handle:
        for record in SeqIO.parse(handle, "fasta"):
            if record.id.strip().lower() in ["1", "chr1"]:
                seq = str(record.seq)
                power = [1] * window_size

                # Precompute powers for rolling hash
                for i in range(1, window_size):
                    power[i] = (power[i - 1] * base) % M

                # Initialize rolling hash
                current_hash = 0
                for i in range(window_size):
                    current_hash = (current_hash * base + ord(seq[i])) % M

                min_hash = current_hash

                for i in range(1, len(seq) - window_size + 1):
                    # Update rolling hash
                    current_hash = (
                        (current_hash - power[window_size - 1] * ord(seq[i - 1])) % M
                    )
                    current_hash = (current_hash * base + ord(seq[i + window_size - 1])) % M

                    # Track min
                    if current_hash < min_hash:
                        min_hash = current_hash

                break  # Only process chr1

    # Normalize and estimate distinct k-mers
    if min_hash is not None:
        normalized_min = min_hash / M
        estimated_distinct_kmers = (1 / normalized_min) - 1
        return estimated_distinct_kmers, normalized_min, min_hash
    else:
        return None, None, None


In [ ]:
# 3c for chromosome 1
# Driver code


minimum_hash = []
all_normalized_min = []

# def main_test_bases():
    window_size = 15
    bases_to_test = [1, 2, 5, 10, 26, 100]
    normalized_mins = []
    estimated_counts =[]
    for base in bases_to_test:
        fasta_file = r"C:\Users\carol\Downloads\human_g1k_v37.fasta.gz"
        hash_values = estimate_distinct_15mers(fasta_file, base=base)
        min_hash = min(hash_values) if hash_values else None
        minimum_hash.append(min_hash)
        M = 10**9 + 7
        normalized_min = min_hash/ M
        all_normalized_min.append(normalized_min)
        print(f"Base {base}: normalized minimum: {normalized_min}, minimum hash {min_hash} All: {hash_values}")
    combined_result = sum(all_normalized_min)/len(all_normalized_min)
    print(f"mean of minima across hash funcations {combined_result}")
    estimated_distinct_kmers = (1 / combined_result) - 1
    print(f"estimate of distinct 15-mers based on all hashes:{estimated_distinct_kmers}")

if __name__ == '__main__':
    main_test_bases()

Base 1: normalized minimum: 9.749999863500002e-16, minimum hash 9.74999993175e-07 All: (1025640.0328205129, 9.74999993175e-07, 975)
Base 2: normalized minimum: 2.1298549701820307e-12, minimum hash 0.0021298549850910153 All: (468.51553368656545, 0.0021298549850910153, 2129855)
Base 5: normalized minimum: 1.5062999789118002e-14, minimum hash 1.5062999894559e-05 All: (66386.83821283942, 1.5062999894559e-05, 15063)
Base 10: normalized minimum: 5.6888883203555634e-11, minimum hash 0.056888883601777816 All: (16.578126633667132, 0.056888883601777816, 56888884)
Base 26: normalized minimum: 7.999999888e-18, minimum hash 7.999999944e-09 All: (124999999.875, 7.999999944e-09, 8)
Base 100: normalized minimum: 8.399999882400002e-17, minimum hash 8.399999941200001e-08 All: (11904760.988095237, 8.399999941200001e-08, 84)
mean of minima across hash funcations 9.839144695585307e-12
estimate of distinct 15-mers based on all hashes:101634850480.33163


In [ ]:
# Initially had mean of minima being calculated across all hash (seen above). Worked with ChatGPT to get minimum/distinct of each base tested
def main_test_bases_1():
    fasta_file = r"C:\Users\carol\Downloads\human_g1k_v37.fasta.gz"
    bases_to_test = [1, 2]

    normalized_mins = []
    estimated_counts = []

    for base in bases_to_test:
        print(f"\n--- Testing with base {base} ---")
        estimate, norm_min, min_hash = estimate_distinct_15mers(fasta_file, base=base)

        if estimate is not None:
            normalized_mins.append(norm_min)
            estimated_counts.append(estimate)

            print(f"Base: {base}")
            print(f"Minimum hash: {min_hash}")
            print(f"Normalized min: {norm_min:.8f}")
            print(f"Estimated distinct 15-mers: {estimate:,.0f}")
        else:
            print(f"No valid hash computed for base {base}")


--- Testing with base 1 ---
Base: 1
Minimum hash: 975
Normalized min: 0.00000097
Estimated distinct 15-mers: 1,025,640

--- Testing with base 2 ---
Base: 2
Minimum hash: 2129855
Normalized min: 0.00212985
Estimated distinct 15-mers: 469


In [ ]:
# Plot the estimated distinct count against the true number of distinct 15-mers for varying numbers of hash functions
# Discuss how estimate improves as more hash functions are combined
# How stable are the estimates? What about only one single hash

# My estimated distinct counts 

In [ ]:
#Justification for design choices 

In [31]:
import gzip
from Bio import SeqIO

def estimate_distinct_15mers_multihash(fasta_file, num_hashes=10, M=10**9 + 7):
    bases = [101 + i*2 for i in range(num_hashes)]  # Ensure different odd bases
    min_hashes = [None] * num_hashes
    window_size = 15

    with gzip.open(fasta_file, "rt") as handle:
        for record in SeqIO.parse(handle, "fasta"):
            if record.id.strip().lower() in ["1", "chr1"]:
                seq = str(record.seq)
                n = len(seq)

                for h, base in enumerate(bases):
                    power = [1] * window_size
                    for i in range(1, window_size):
                        power[i] = (power[i - 1] * base) % M

                    current_hash = 0
                    for i in range(window_size):
                        current_hash = (current_hash * base + ord(seq[i])) % M

                    min_hash = current_hash

                    for i in range(1, n - window_size + 1):
                        if 'N' in seq[i - 1:i + window_size]:  # Skip if 'N' in 15-mer
                            continue
                        current_hash = (
                            (current_hash - power[window_size - 1] * ord(seq[i - 1])) % M
                        )
                        current_hash = (current_hash * base + ord(seq[i + window_size - 1])) % M
                        if min_hash is None or current_hash < min_hash:
                            min_hash = current_hash

                    min_hashes[h] = min_hash

                break  # Only chromosome 1

    # Normalize and estimate
    normalized_mins = [h / M for h in min_hashes if h is not None]
    mean_min = sum(normalized_mins) / len(normalized_mins)
    estimated_distinct = (1 / mean_min) - 1 if mean_min > 0 else 0

    return estimated_distinct, normalized_mins, min_hashes, mean_min


In [34]:
hash_counts = [1, 2]
estimates = []

for n_hash in hash_counts:
    est, _, _, _ = estimate_distinct_15mers_multihash(fasta_file, num_hashes=n_hash)
    print(f"{n_hash} hash(es): estimated = {int(est)}")
    estimates.append(est)

1 hash(es): estimated = 333333334
2 hash(es): estimated = 222222222
